# Lab 1 — Triage Agent (🟡 Builder rail)

> 🟡 **Builder rail.** Run the cells top to bottom. The lines marked `# 👉` are the parts you wire up; the `# TODO (bonus)` cell is optional. Same checkpoint as the other rails.

**Goal:** make Care Pal return a structured **7‑key triage JSON** on every turn and route each message by clinical risk.
Run from the `assets/` folder (so `common` imports resolve): `python lab1_triage.py`

### New to agents? Here's the tech stack in plain terms

Think of it like calling a model API — but instead of crafting a prompt each time, you first **register an "agent"** (a reusable model + instructions + output contract) and then send it messages.

| Building block | What it is | Why it matters here |
|----------------|-----------|---------------------|
| **Microsoft Foundry Agent Service** | The managed backend that hosts your agent. The **same** service the no‑code Foundry portal uses. | Your notebook agent and a portal agent are the *same object* — just created two different ways. |
| **`azure-ai-projects` (v2.x) SDK** | The Python library you talk to Foundry with. | Provides `project.agents.create_version(...)` to **define and version** your agent. |
| **`PromptAgentDefinition`** | The object that describes your agent: which **model**, what **instructions**, and its **output format**. | This *is* your agent's configuration — the code equivalent of the portal's setup screen. |
| **Structured output** (`PromptAgentDefinition.text` + a JSON schema) | A rule that forces the model to reply with a **fixed JSON shape**, not free text. | Guarantees the 7 keys every turn, so downstream software can act on the result. |
| **OpenAI‑compatible client** | `responses.create(...)` — the familiar OpenAI‑style call, pointed at Foundry. | If you've used the OpenAI SDK, running the agent will feel identical. |

> 📚 **Where these patterns come from (optional reading):** the official Microsoft samples — the *prompt‑agent quickstart* (shows structured output via `PromptAgentDefinition.text`) and `sample_agent_structured_output.py` in the `azure-ai-projects` 2.x samples. You don't need to open them; the `common/` helpers already apply these patterns for you.


## Cell 1 — Make the shared helpers importable

Before we can talk to Foundry, Python needs to find the `common/` helper package that sits next to this notebook. This cell adds the notebook's folder (`assets/`) to Python's import path.

- It works whether you **run this as a notebook** (uses the current working directory) or **as a script** (`python lab1_triage.py`, uses the file's location).
- Nothing touches Microsoft Foundry yet — this is pure local setup so the next cell's `from common.carepal_common import ...` succeeds.


In [1]:
# --- make `common` importable whether run as a script or a notebook ---
import sys
import pathlib
_here = (pathlib.Path(globals()["__file__"]).resolve().parent
         if "__file__" in globals() else pathlib.Path.cwd())
if str(_here) not in sys.path:
    sys.path.insert(0, str(_here))

## Cell 2 — Import the Care Pal helpers and pick the test cases

This cell pulls in the helper functions that wrap the Microsoft Foundry Agent Service so the lab stays short. Each import maps to one step of the triage flow:

| Helper | What it does with Foundry |
|--------|---------------------------|
| `make_triage_agent` | Calls `project.agents.create_version(...)` to **create (or re-version) your `carepal-jb` agent** in the shared Foundry project — model + triage instructions + the 7‑key JSON schema. |
| `run_and_parse` | Sends one user message to the agent via the OpenAI‑compatible client (`responses.create(... agent_reference)`) and parses the JSON it returns. |
| `text_of` / `route_of` | Look up the verbatim test message and its **expected** route from `../prompts/test-prompts.json` (the single source of truth shared by every rail). |
| `REQUIRED_KEYS` | The 7 keys every triage reply must contain (`intent`, `risk_level`, `route`, `reply`, `source_labels`, `source_urls`, `clarifying_questions`). |
| `cleanup` | Deletes the agent afterwards so the shared project stays tidy. |

`CASES` is the list of three prompt IDs we'll route in this lab: a stable diet question, a worsening‑symptom report, and a red‑flag chest‑pain message.


In [2]:
from common.carepal_common import (
    make_triage_agent,
    run_and_parse,
    text_of,
    route_of,
    REQUIRED_KEYS,
    cleanup,
)

# The three messages we route in this lab (see ../prompts/test-prompts.json).
CASES = ["diet_question", "swelling_worsening", "chest_pain"]

## Cell 3 — Create the agent, route the three messages, and check the results

This is the heart of the lab. When you run it, it does the full round‑trip with Microsoft Foundry:

1. **Create the agent** — `make_triage_agent(structured=True)` provisions `carepal-jb` in the shared Foundry project. Passing `structured=True` attaches a **JSON‑schema structured‑output format** to the agent definition, so Foundry forces the model to return exactly the 7 keys every turn (no stray prose).
2. **Loop the test cases** — for each prompt ID, `run_and_parse` sends the message to Foundry's Agent Service and parses the JSON reply.
3. **Validate two things per case:**
   - **Completeness** — all 7 `REQUIRED_KEYS` are present (the `assert not missing` line).
   - **Correct routing** — the agent's `route` matches the expected route from `test-prompts.json` (`out["route"] == route_of(pid)`).
4. **Print a pass line** for each case, then `Lab 1 passed ✅` when all three succeed.
5. The agent is **left running on purpose** so you can inspect it in the portal (Cell 4). You delete it yourself in the final cleanup cell when you're done.

**Expected output:**
```
Created agent: carepal-jb (version 1)
OK  diet_question      -> education_navigation
OK  swelling_worsening -> timely_review
OK  chest_pain         -> immediate_escalation
Lab 1 passed ✅
```

> 💡 The `# TODO (bonus)` note is optional: add a `medication_question` intent to the instructions and assert a medication‑timing question routes to `timely_review` to earn the bonus badge.


In [3]:
# 👉 Create the triage agent. structured=True enforces the 7-key JSON contract.
# `agent` is kept as a top-level variable so you can inspect it in the Foundry
# portal and then delete it from the separate cleanup cell below.
agent = make_triage_agent(structured=True)
print(f"Created agent: {agent.name} (version {agent.version})")

for pid in CASES:
    out = run_and_parse(agent, text_of(pid))
    missing = set(REQUIRED_KEYS) - set(out)
    assert not missing, f"{pid}: missing keys {missing}"
    assert out["route"] == route_of(pid), f"{pid}: {out['route']} != {route_of(pid)}"
    print(f"OK  {pid:18} -> {out['route']}")
print("Lab 1 passed ✅")

# TODO (bonus): add intent 'medication_question' to your instructions and assert that a
# medication-timing question routes to 'timely_review'. (Earns the Schema Surgeon badge.)


Created agent: carepal-jb (version 4)
OK  diet_question      -> education_navigation
OK  swelling_worsening -> timely_review
OK  chest_pain         -> immediate_escalation
Lab 1 passed ✅


## Cell 4 — Inspect your agent in the Foundry portal (optional pause)

Your `carepal-jb` agent is now **live in the shared Foundry project** — the cleanup step is intentionally in the *next* cell so you can go look at it first.

1. Open **https://ai.azure.com** → project **`research-workshop`** → **Agents**.
2. Find **`carepal-jb`** and open it. Notice it has the **same instructions and JSON output format** you set from code — because the notebook and the portal drive the *same* Agent Service.
3. Try it in the **Playground** with the chest‑pain message to see the structured JSON come back.

> ℹ️ **No conflict if you re-run:** re-running the cell above simply creates a **new version** of `carepal-jb` (Foundry versions agents by name). Your name is unique in the shared project, so you never clash with other participants.


In [4]:
# 🧹 Run this LAST — once you've finished inspecting the agent in the portal.
# Deletes the version you created so the shared project stays tidy. Safe to run
# even if you re-created the agent a few times; it removes the current version.
cleanup(agent)
print(f"Cleaned up {agent.name} ✅")


Cleaned up carepal-jb ✅
